In [1]:
import requests

api_url = "https://115.159.159.82/api/devices"

r = requests.get(api_url, params={"page": 1, "page_size": 20}, timeout=10)
print(r.status_code, r.url)
data = r.json()
print(data.keys())          # dict_keys(['page','page_size','total','items'])
print(data["page"], data["page_size"], data["total"])   # 1 20 5000
print(data["items"][0])

200 https://115.159.159.82/api/devices?page=1&page_size=20
dict_keys(['page', 'page_size', 'total', 'items'])
1 20 5000
{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [5]:
total = data["total"]       
page_size = data["page_size"]   
total_pages = (total + page_size - 1) // page_size
print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 20
total_pages: 250


In [7]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
           "page":page,
            "page_size":100
        },
        timeout=10
    )

    page_data = r.json()

    # 将本页设备加入 devices
    # TODO
    devices.extend(r.json()["items"])
print("device count:", len(devices))

device count: 5000


In [8]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [9]:
cards = soup.select("article.company-card")
print("本页企业数：", len(cards))

本页企业数： 20


In [12]:
card = cards[0]

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value
# {'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}

company = {
    "company_id": card.get("data-company-id"),                             
    "name": card.select_one("h2 a").get_text(strip=True),                  
    "industry": details["所属行业"],
    "zone": details["所在区域"],
    "device_count": int(details["设备数量"].replace("台", "").strip()),     
    "status": card.select_one("span.badge").get_text(strip=True),         
}
print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [13]:
next_link = soup.select_one("填写你设计的下一页 selector")
next_link = soup.select_one('nav.pagination a:-soup-contains("下一页")')
print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [14]:
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

https://115.159.159.82/companies?page=2


In [15]:
from urllib.parse import urljoin

base_url = "https://115.159.159.82"
url = base_url + "/companies"
companies = []

for _ in range(3):                       # 只采 3 页
    r = requests.get(url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    for card in soup.select("article.company-card"):
        ...                              # 用 4.3 的字段提取逻辑

    nxt = soup.select_one('nav.pagination a:-soup-contains("下一页")')
    if not nxt:
        break
    url = urljoin(url, nxt.get("href"))   # href 是相对路径，必须 urljoin

print(len(companies))    # 60

0


In [16]:
from collections import Counter

print("companies:", len(companies), "unique:", len(set(x["company_id"] for x in companies)))
# companies: 60 unique: 60

print("devices:", len(devices), "unique:", len(set(x["device_id"] for x in devices)))
# devices: 5000 unique: 5000

device_counts = Counter(x["company_id"] for x in devices)
page_counts = {c["company_id"]: c["device_count"] for c in companies}

mismatch = [(k, device_counts.get(k, 0), v) for k, v in page_counts.items() if device_counts.get(k, 0) != v]
print("不一致企业数:", len(mismatch))     # 0

companies: 0 unique: 0
devices: 5000 unique: 5000
不一致企业数: 0


In [18]:
from pathlib import Path
import json

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
